# Bronze to Silver — CineData Analytics
Limpeza, padronização, tradução e deduplicação das tabelas Bronze, seguindo as regras de
negócio da seção 1.3 do enunciado. Nenhuma tabela Bronze é alterada.

In [ ]:
import os
import sys

sys.path.append(os.path.abspath(".."))

from src.silver.avaliacoes import transformar_avaliacoes_usuarios
from src.silver.cotacao_dolar import obter_cotacao_mais_recente, transformar_cotacao_dolar
from src.silver.engajamento import transformar_metricas_engajamento
from src.silver.filmes import transformar_info_filmes
from src.silver.financeiro import transformar_financeiro_filmes
from src.silver.generos import transformar_generos
from src.silver.pessoas_empresas import unificar_pessoas_empresas

# Widgets sao por notebook: declarar aqui tambem (mesmos defaults do Landing_to_Bronze),
# senao o Job falha ao ler data_inicio/data_fim.
from datetime import date, timedelta

dbutils.widgets.text("data_inicio", (date.today() - timedelta(days=7)).isoformat())
dbutils.widgets.text("data_fim", date.today().isoformat())

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

**Regras de `silver.tb_info_filmes`:**
- **Status:** normaliza (tira ruído, hífens e caixa) *antes* de traduzir; o que não mapear vira "Não Informado". Traduzir antes deixaria "released-" sem casar com "Released".
- **Deduplicação:** o mesmo filme aparece até 34 vezes na origem; mantém o de `ingestion_datetime` mais recente e, no empate, o mais completo (com desempate por hash, para o resultado ser determinístico).
- **Datas:** a origem mistura `yyyy-MM-dd`, `MM-dd-yyyy` e `dd/MM/yyyy`; testa cada formato e só vira `NULL` o que for impossível de converter.
- **Tipagem:** `duracao_minutos` vira inteiro por conversão segura (texto vazado vira `NULL`, sem derrubar o pipeline); `ano_lancamento` é derivado da data.

In [ ]:
# 1) silver.tb_info_filmes
df_info = spark.table("bronze.tb_movies_info")
df_filmes = transformar_info_filmes(df_info)
df_filmes.write.format("delta").mode("overwrite").saveAsTable("silver.tb_info_filmes")
display(df_filmes)

**Regras de `silver.tb_metricas_engajamento`:**
- **Vírgula decimal** (`89,985`) vira ponto; nenhum outro caractere é apagado (apagar juntava dígitos de texto vazado e criava números falsos).
- **Column shift:** texto em coluna numérica vira `NULL` por conversão segura, sem interromper o pipeline.
- **Limites de negócio:** nota fora de 0 a 10 (inclui erro de escala, ex.: 63.65) e contagem/popularidade negativa viram `NULL`.
- **Deduplicação por filme**, como na tabela de filmes.

In [ ]:
# 3) silver.tb_metricas_engajamento
df_metrics = spark.table("bronze.tb_movies_metrics")
df_engajamento = transformar_metricas_engajamento(df_metrics)
df_engajamento.write.format("delta").mode("overwrite").saveAsTable("silver.tb_metricas_engajamento")
display(df_engajamento)

**Regras de `silver.tb_avaliacoes_usuarios`:** remove duplicatas exatas (filme + usuário + nota + comentário); nota fora de 0 a 10 vira `NULL`; comentário vazio ou só com espaços vira "Sem comentário".

In [ ]:
# 4) silver.tb_avaliacoes_usuarios
df_reviews = spark.table("bronze.tb_movies_reviews")
df_avaliacoes = transformar_avaliacoes_usuarios(df_reviews)
df_avaliacoes.write.format("delta").mode("overwrite").saveAsTable("silver.tb_avaliacoes_usuarios")
display(df_avaliacoes)

**Regras de `silver.tb_generos` e `silver.tb_pessoas_empresas`** (mesma origem, `tb_credits_and_tags`):
- **Separadores:** vírgula, ponto e vírgula e `|` são tratados como separadores antes do split e do explode.
- **Gêneros:** só valores do domínio de 19 gêneros do catálogo; sinopses, caminhos de imagem e números deslocados (column shift) são descartados.
- **Pessoas e empresas:** unifica ator, diretor, roteirista e produtora em `tipo_entidade`; limpa aspas e barras nas pontas; descarta números, caminhos de imagem, frases longas e nomes de 1 caractere; padroniza a caixa preservando nomes mistos (O'Reilly, DiCaprio); remove duplicatas.

In [ ]:
# 5) silver.tb_generos e 6) silver.tb_pessoas_empresas (mesma origem: tb_credits_and_tags)
df_credits = spark.table("bronze.tb_credits_and_tags")

df_generos = transformar_generos(df_credits)
df_generos.write.format("delta").mode("overwrite").saveAsTable("silver.tb_generos")

df_pessoas_empresas = unificar_pessoas_empresas(df_credits)
df_pessoas_empresas.write.format("delta").mode("overwrite").saveAsTable("silver.tb_pessoas_empresas")

display(df_pessoas_empresas)

**Regras de `silver.tb_cotacao_dolar`:** gera um calendário diário contínuo e aplica *forward fill* (dias sem cotação recebem o último valor útil). A cotação mais recente alimenta a conversão para BRL, por isso esta célula roda **antes** do financeiro.

In [ ]:
# 7) silver.tb_cotacao_dolar (forward-fill) -- precisa rodar antes do financeiro,
# que depende da cotacao mais recente calculada aqui.
df_cotacao_bronze = spark.table("bronze.tb_cotacao_dolar")
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

df_cotacao_silver = transformar_cotacao_dolar(df_cotacao_bronze, spark, data_inicio, data_fim)
df_cotacao_silver.write.format("delta").mode("overwrite").saveAsTable("silver.tb_cotacao_dolar")

cotacao_atual = obter_cotacao_mais_recente(df_cotacao_silver)
print(f"Cotação usada para conversão BRL: {cotacao_atual}")

**Regras de `silver.tb_financeiro_filmes`:**
- **Valores monetários:** interpreta `97000000`, `$ 97000000`, `USD 150000000` e `34.0M`/`250.5K` (34 milhões, não 34); `Unknown`, `N/A` e texto vazado viram `NULL`; valor que não cabe em `decimal(18,2)` também vira `NULL`, em vez de derrubar o pipeline.
- **Zero e negativo viram `NULL`:** ~90% dos orçamentos e receitas são `0` na origem, ou seja, ausência de dado.
- **BRL:** aplica a cotação mais recente. **Lucro e margem** ficam `NULL` quando falta um dos valores e nunca dividem por zero.
- **Deduplicação por filme**, para a fato ter um registro por filme.

In [ ]:
# 2) silver.tb_financeiro_filmes (depende da cotacao calculada na celula anterior)
df_financials = spark.table("bronze.tb_movies_financials")
df_financeiro = transformar_financeiro_filmes(df_financials, cotacao=cotacao_atual)
df_financeiro.write.format("delta").mode("overwrite").saveAsTable("silver.tb_financeiro_filmes")
display(df_financeiro)